# Step 1: Installation and Setup

In [ ]:
# Installing TensorFlow
! pip install -q tensorflow-gpu

In [ ]:
import tensorflow as tf

In [ ]:
print(tf.__version__)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Step 2: Importing the dataset from kaggle to google colab

In [ ]:
# install kaggle API
! pip install kaggle

In [ ]:
# create a directory as kaggle
! mkdir -p ~/.kaggle

In [ ]:
# import kaggle API
from google.colab import files

uploaded = files.upload()

In [ ]:
# copy API key to kaggle directory
! cp kaggle.json ~/.kaggle

In [ ]:
# disable the API key
! chmod 600 /root/.kaggle/kaggle.json

In [ ]:
# list of datasets
! kaggle datasets list

In [ ]:
# import the dataset
! kaggle datasets download -d mlg-ulb/creditcardfraud

In [ ]:
# unzipping dataset
! unzip /content/creditcardfraud.zip

In [ ]:
dataset_1 = pd.read_csv("/content/creditcard.csv")

In [ ]:
dataset_1.head()

# Step 3: Data Preprocessing

In [ ]:
dataset_1.shape

In [ ]:
# checking the null values
dataset_1.isnull().sum()

In [ ]:
dataset_1.info()

In [ ]:
# observations in each class
dataset_1["Class"].value_counts()

In [ ]:
# balence the dataset
fraud = dataset_1[dataset_1["Class"] == 1]
non_fraud = dataset_1[dataset_1["Class"] == 0]

In [ ]:
fraud.shape, non_fraud.shape

In [ ]:
# random selection of samples
non_fraud_t = non_fraud.sample(n=492)

In [ ]:
non_fraud_t.shape

In [ ]:
# merge dataset
dataset = fraud.append(non_fraud_t, ignore_index=True)

In [ ]:
print(dataset)

In [ ]:
# observations in each class
dataset["Class"].value_counts()

In [ ]:
# matrix of features
x = dataset.drop(labels=["Class"], axis=1)

In [ ]:
# dependent variable
y = dataset["Class"]

In [ ]:
x.shape, y.shape

In [ ]:
# splitting the dataset into train and test set
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

In [ ]:
x_train.shape, x_test.shape

In [ ]:
# feature scaling
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [ ]:
x_train

In [ ]:
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [ ]:
x_train.shape, x_test.shape

In [ ]:
# reshape the dataset
x_train = x_train.reshape(787, 30, 1)
x_test = x_test.reshape(197, 30, 1)

In [ ]:
x_train.shape, x_test.shape

# Step 4: Building the model

In [ ]:
# defining an object
model = tf.keras.models.Sequential()

In [ ]:
# first CNN layer
model.add(
    tf.keras.layers.Conv1D(
        filters=32,
        kernel_size=2,
        padding="same",
        activation="relu",
        input_shape=(30, 1),
    )
)

# batch normalization
model.add(tf.keras.layers.BatchNormalization())

# maxpool layer
model.add(tf.keras.layers.MaxPool1D(pool_size=2))

# dropout layer
model.add(tf.keras.layers.Dropout(0.2))

In [ ]:
# second CNN layer
model.add(
    tf.keras.layers.Conv1D(filters=64, kernel_size=2, padding="same", activation="relu")
)

# batch normalization
model.add(tf.keras.layers.BatchNormalization())

# maxpool layer
model.add(tf.keras.layers.MaxPool1D(pool_size=2))

# dropout layer
model.add(tf.keras.layers.Dropout(0.3))

In [ ]:
# flatten layer
model.add(tf.keras.layers.Flatten())

In [ ]:
# first dense layer
model.add(tf.keras.layers.Dense(units=64, activation="relu"))

# dropout layer
model.add(tf.keras.layers.Dropout(0.3))

In [ ]:
# output layer
model.add(tf.keras.layers.Dense(units=1, activation="sigmoid"))

In [ ]:
model.summary()

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.0001)

In [ ]:
model.compile(optimizer=opt, loss="binary_crossentropy", metrics=["accuracy"])

# Step 5: Training the model

In [ ]:
history = model.fit(x_train, y_train, epochs=25, validation_data=(x_test, y_test))

In [ ]:
# model predictions
y_pred = model.predict_classes(x_test)

In [ ]:
print(y_pred[12]), print(y_test[12])

In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix, accuracy_score

cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
acc_cm = accuracy_score(y_test, y_pred)
print(acc_cm)

# Step 6: Learning Curve

In [ ]:
def learning_curve(history, epoch):

    # training vs validation accuracy
    epoch_range = range(1, epoch + 1)
    plt.plot(epoch_range, history.history["accuracy"])
    plt.plot(epoch_range, history.history["val_accuracy"])
    plt.title("Model Accuracy")
    plt.ylabel("Accuracy")
    plt.xlabel("Epoch")
    plt.legend(["Train", "val"], loc="upper left")
    plt.show()

    # training vs validation loss
    plt.plot(epoch_range, history.history["loss"])
    plt.plot(epoch_range, history.history["val_loss"])
    plt.title("Model Loss")
    plt.ylabel("Loss")
    plt.xlabel("Epoch")
    plt.legend(["Train", "val"], loc="upper left")
    plt.show()

In [ ]:
learning_curve(history, 25)